In [ ]:
from datetime import datetime, timedelta
from airflow.decorators import dag, task
import os

# Base directory where Airflow is running inside the container.
BASE_PATH = "/opt/airflow"

# Full path to the DuckDB database file.
DB_PATH = os.path.join(BASE_PATH, "dw_v2.duckdb")

# Directory where the CSV files arrive.
STAGING_DIR = os.path.join(BASE_PATH, "staging")

# Default Airflow arguments for all tasks in this DAG.
default_args = {
    "owner": "data_engineer",
    "retries": 1,
    "retry_delay": timedelta(minutes=2),
}


@dag(
    dag_id="elt_duckdb_pipeline_V3",
    default_args=default_args,
    schedule="@daily",
    start_date=datetime(2025, 1, 1),
    catchup=False,
)
def elt_pipeline():
    """
    ELT pipeline with incremental loading logic.

    General flow:
    1. Prepare control structures and clean current staging table.
    2. Detect new or modified CSV files in staging.
    3. Load only those new/changed files into staging_raw.
    4. Transform staging data and merge it into the historical fact table.
    5. Calculate final metrics for monitoring.
    """

    @task()
    def preparar_control_y_staging():
        """
        Prepares the technical environment inside DuckDB for this execution.

        What it does:
        - Ensures the control table processed_files exists.
        - Drops staging_raw so each run starts with a clean staging area.
        """
        import duckdb
        import logging

        conn = duckdb.connect(DB_PATH)

        conn.execute("""
            CREATE TABLE IF NOT EXISTS processed_files (
                file_name VARCHAR,
                file_path VARCHAR,
                file_size_bytes BIGINT,
                file_modified_at TIMESTAMP,
                processed_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        conn.execute("DROP TABLE IF EXISTS staging_raw")

        logging.info("Tabla staging_raw eliminada para la ejecución actual")
        logging.info("Tabla processed_files verificada")

        conn.close()

    @task()
    def cargar_staging_incremental():
        """
        Loads only new or modified CSV files into staging_raw.

        Incremental rule:
        A file is considered already processed if the following three values match
        an existing row in processed_files:
        - file_name
        - file_size_bytes
        - file_modified_at

        If any of those values changed, the file is treated as new/modified
        and will be reprocessed.
        """
        import duckdb
        import logging
        import glob
        import os
        from datetime import datetime

        conn = duckdb.connect(DB_PATH)

        csv_files = sorted(glob.glob(os.path.join(STAGING_DIR, "*.csv")))

        if not csv_files:
            logging.info("No se encontraron archivos CSV en staging")
            conn.execute("CREATE TABLE staging_raw AS SELECT * FROM (SELECT 1 AS dummy) WHERE 1=0")
            conn.close()

            return {
                "files_found": 0,
                "files_to_process": 0,
                "rows_loaded": 0,
                "processed_files": []
            }

        files_to_process = []

        for file_path in csv_files:
            stat = os.stat(file_path)
            file_name = os.path.basename(file_path)
            file_size_bytes = stat.st_size
            file_modified_at = datetime.fromtimestamp(stat.st_mtime)

            exists = conn.execute(
                """
                SELECT 1
                FROM processed_files
                WHERE file_name = ?
                  AND file_size_bytes = ?
                  AND file_modified_at = ?
                LIMIT 1
                """,
                [file_name, file_size_bytes, file_modified_at],
            ).fetchone()

            if exists is None:
                files_to_process.append(
                    {
                        "file_name": file_name,
                        "file_path": file_path,
                        "file_size_bytes": file_size_bytes,
                        "file_modified_at": file_modified_at,
                    }
                )

        if not files_to_process:
            logging.info("Todos los archivos ya fueron procesados previamente. No se cargará información nueva.")
            conn.execute("CREATE TABLE staging_raw AS SELECT * FROM (SELECT 1 AS dummy) WHERE 1=0")
            conn.close()

            return {
                "files_found": len(csv_files),
                "files_to_process": 0,
                "rows_loaded": 0,
                "processed_files": []
            }

        select_statements = []

        for file_info in files_to_process:
            safe_path = file_info["file_path"].replace("'", "''")
            safe_name = file_info["file_name"].replace("'", "''")

            select_statements.append(
                f"""
                SELECT
                    *,
                    '{safe_name}' AS source_file
                FROM read_csv_auto('{safe_path}')
                """
            )

        union_sql = "\nUNION ALL\n".join(select_statements)

        conn.execute(f"CREATE TABLE staging_raw AS {union_sql}")

        rows_loaded = conn.execute("SELECT COUNT(*) FROM staging_raw").fetchone()[0]

        for file_info in files_to_process:
            conn.execute(
                """
                INSERT INTO processed_files (
                    file_name,
                    file_path,
                    file_size_bytes,
                    file_modified_at,
                    processed_at
                ) VALUES (?, ?, ?, ?, CURRENT_TIMESTAMP)
                """,
                [
                    file_info["file_name"],
                    file_info["file_path"],
                    file_info["file_size_bytes"],
                    file_info["file_modified_at"],
                ],
            )

        logging.info(f"Archivos encontrados: {len(csv_files)}")
        logging.info(f"Archivos nuevos o modificados a procesar: {len(files_to_process)}")
        logging.info(f"Registros cargados en staging_raw: {rows_loaded}")
        logging.info(
            "Archivos procesados en esta ejecución: %s",
            ", ".join([f["file_name"] for f in files_to_process])
        )

        conn.close()

        return {
            "files_found": len(csv_files),
            "files_to_process": len(files_to_process),
            "rows_loaded": rows_loaded,
            "processed_files": [f["file_name"] for f in files_to_process],
        }

    @task()
    def transformar_datos(staging_info: dict):
        """
        Transforms data from staging_raw and merges it into the historical fact table.

        Key ideas:
        - If there are no new files, do nothing.
        - fact_finanzas_elt is cumulative: it stores historical results.
        - New rows are inserted.
        - Existing rows are updated if their values changed.

        Business key used for matching historical rows:
        - id
        - fecha
        """
        import duckdb
        import logging

        conn = duckdb.connect(DB_PATH)

        if staging_info.get("files_to_process", 0) == 0:
            logging.info("No hay archivos nuevos para transformar. Se conserva el histórico sin cambios.")
            conn.close()
            return

        total_staging = conn.execute("SELECT COUNT(*) FROM staging_raw").fetchone()[0]

        conn.execute("""
            CREATE TABLE IF NOT EXISTS fact_finanzas_elt (
                id BIGINT,
                salario DOUBLE,
                gastos DOUBLE,
                fecha DATE,
                correo_hash VARCHAR,
                utilidad DOUBLE,
                source_file VARCHAR,
                fecha_carga TIMESTAMP DEFAULT CURRENT_TIMESTAMP,
                fecha_actualizacion TIMESTAMP DEFAULT CURRENT_TIMESTAMP
            )
        """)

        conn.execute("DROP TABLE IF EXISTS tmp_fact_finanzas_elt")

        conn.execute("""
            CREATE TEMP TABLE tmp_fact_finanzas_elt AS
            WITH base AS (
                SELECT
                    id,
                    salario,
                    COALESCE(gastos, 0) AS gastos,
                    CASE
                        WHEN regexp_matches(CAST(fecha AS VARCHAR), '^[0-9]{4}-[0-9]{2}-[0-9]{2}$')
                            THEN CAST(CAST(fecha AS VARCHAR) AS DATE)
                        WHEN regexp_matches(CAST(fecha AS VARCHAR), '^[0-9]{2}/[0-9]{2}/[0-9]{4}$')
                            THEN CAST(STRPTIME(CAST(fecha AS VARCHAR), '%d/%m/%Y') AS DATE)
                        WHEN regexp_matches(CAST(fecha AS VARCHAR), '^[0-9]{2}-[0-9]{2}-[0-9]{4}$')
                            THEN CAST(STRPTIME(CAST(fecha AS VARCHAR), '%m-%d-%Y') AS DATE)
                        ELSE NULL
                    END AS fecha,
                    CASE
                        WHEN correo IS NOT NULL THEN sha256(CAST(correo AS VARCHAR))
                        ELSE NULL
                    END AS correo_hash,
                    source_file,
                    CURRENT_TIMESTAMP AS fecha_proceso
                FROM staging_raw
            ),
            filtrado AS (
                SELECT
                    id,
                    salario,
                    gastos,
                    fecha,
                    correo_hash,
                    salario - gastos AS utilidad,
                    source_file,
                    fecha_proceso AS fecha_carga,
                    fecha_proceso AS fecha_actualizacion
                FROM base
                WHERE
                    id IS NOT NULL
                    AND fecha IS NOT NULL
                    AND salario > 0
                    AND gastos >= 0
            ),
            deduplicado AS (
                SELECT *
                FROM (
                    SELECT
                        *,
                        ROW_NUMBER() OVER (
                            PARTITION BY id, fecha
                            ORDER BY fecha_actualizacion DESC, source_file DESC
                        ) AS rn
                    FROM filtrado
                )
                WHERE rn = 1
            )
            SELECT
                id,
                salario,
                gastos,
                fecha,
                correo_hash,
                utilidad,
                source_file,
                fecha_carga,
                fecha_actualizacion
            FROM deduplicado
        """)

        total_transformados = conn.execute(
            "SELECT COUNT(*) FROM tmp_fact_finanzas_elt"
        ).fetchone()[0]

        nuevos = conn.execute("""
            SELECT COUNT(*)
            FROM tmp_fact_finanzas_elt s
            LEFT JOIN fact_finanzas_elt t
              ON t.id = s.id
             AND t.fecha = s.fecha
            WHERE t.id IS NULL
        """).fetchone()[0]

        actualizaciones = conn.execute("""
            SELECT COUNT(*)
            FROM tmp_fact_finanzas_elt s
            JOIN fact_finanzas_elt t
              ON t.id = s.id
             AND t.fecha = s.fecha
            WHERE
                COALESCE(t.salario, -1) <> COALESCE(s.salario, -1)
                OR COALESCE(t.gastos, -1) <> COALESCE(s.gastos, -1)
                OR COALESCE(t.correo_hash, '') <> COALESCE(s.correo_hash, '')
                OR COALESCE(t.utilidad, -1) <> COALESCE(s.utilidad, -1)
                OR COALESCE(t.source_file, '') <> COALESCE(s.source_file, '')
        """).fetchone()[0]

        conn.execute("""
            UPDATE fact_finanzas_elt AS target
            SET
                salario = source.salario,
                gastos = source.gastos,
                correo_hash = source.correo_hash,
                utilidad = source.utilidad,
                source_file = source.source_file,
                fecha_actualizacion = CURRENT_TIMESTAMP
            FROM tmp_fact_finanzas_elt AS source
            WHERE target.id = source.id
              AND target.fecha = source.fecha
              AND (
                    COALESCE(target.salario, -1) <> COALESCE(source.salario, -1)
                    OR COALESCE(target.gastos, -1) <> COALESCE(source.gastos, -1)
                    OR COALESCE(target.correo_hash, '') <> COALESCE(source.correo_hash, '')
                    OR COALESCE(target.utilidad, -1) <> COALESCE(source.utilidad, -1)
                    OR COALESCE(target.source_file, '') <> COALESCE(source.source_file, '')
              )
        """)

        conn.execute("""
            INSERT INTO fact_finanzas_elt (
                id,
                salario,
                gastos,
                fecha,
                correo_hash,
                utilidad,
                source_file,
                fecha_carga,
                fecha_actualizacion
            )
            SELECT
                s.id,
                s.salario,
                s.gastos,
                s.fecha,
                s.correo_hash,
                s.utilidad,
                s.source_file,
                s.fecha_carga,
                s.fecha_actualizacion
            FROM tmp_fact_finanzas_elt s
            LEFT JOIN fact_finanzas_elt t
              ON t.id = s.id
             AND t.fecha = s.fecha
            WHERE t.id IS NULL
        """)

        total_final = conn.execute(
            "SELECT COUNT(*) FROM fact_finanzas_elt"
        ).fetchone()[0]

        logging.info(f"Registros leídos en staging: {total_staging}")
        logging.info(f"Registros válidos transformados: {total_transformados}")
        logging.info(f"Registros nuevos insertados: {nuevos}")
        logging.info(f"Registros actualizados: {actualizaciones}")
        logging.info(f"Total histórico en fact_finanzas_elt: {total_final}")
        logging.info("Carga incremental completada sin eliminar histórico")

        conn.close()

    @task()
    def metricas_finales():
        """
        Calculates monitoring metrics from the historical fact table
        and prints them clearly in Airflow logs.
        """
        import duckdb
        import logging

        conn = duckdb.connect(DB_PATH)

        existe = conn.execute("""
            SELECT COUNT(*)
            FROM information_schema.tables
            WHERE table_name = 'fact_finanzas_elt'
        """).fetchone()[0]

        if existe == 0:
            logging.info("La tabla fact_finanzas_elt no existe todavía.")
            conn.close()
            return {
                "total_registros": 0,
                "total_ids": 0,
                "total_fechas": 0,
                "promedio_utilidad": 0,
                "max_utilidad": 0,
                "min_utilidad": 0,
                "suma_utilidad": 0,
            }

        resultado = conn.execute("""
            SELECT
                COUNT(*) AS total_registros,
                COUNT(DISTINCT id) AS total_ids,
                COUNT(DISTINCT fecha) AS total_fechas,
                AVG(utilidad) AS promedio_utilidad,
                MAX(utilidad) AS max_utilidad,
                MIN(utilidad) AS min_utilidad,
                SUM(utilidad) AS suma_utilidad
            FROM fact_finanzas_elt
        """).fetchone()

        conn.close()

        (
            total_registros,
            total_ids,
            total_fechas,
            promedio_utilidad,
            max_utilidad,
            min_utilidad,
            suma_utilidad,
        ) = resultado

        promedio_utilidad = round(promedio_utilidad, 2) if promedio_utilidad is not None else 0
        max_utilidad = round(max_utilidad, 2) if max_utilidad is not None else 0
        min_utilidad = round(min_utilidad, 2) if min_utilidad is not None else 0
        suma_utilidad = round(suma_utilidad, 2) if suma_utilidad is not None else 0

        logging.info("========== RESUMEN HISTÓRICO ==========")
        logging.info(f"Total de registros    : {total_registros}")
        logging.info(f"IDs distintos         : {total_ids}")
        logging.info(f"Fechas distintas      : {total_fechas}")
        logging.info(f"Promedio utilidad     : {promedio_utilidad}")
        logging.info(f"Máxima utilidad       : {max_utilidad}")
        logging.info(f"Mínima utilidad       : {min_utilidad}")
        logging.info(f"Suma utilidad         : {suma_utilidad}")
        logging.info("=======================================")

        return {
            "total_registros": total_registros,
            "total_ids": total_ids,
            "total_fechas": total_fechas,
            "promedio_utilidad": promedio_utilidad,
            "max_utilidad": max_utilidad,
            "min_utilidad": min_utilidad,
            "suma_utilidad": suma_utilidad,
        }

    preparar = preparar_control_y_staging()
    staging_info = cargar_staging_incremental()
    transformacion = transformar_datos(staging_info)
    metricas = metricas_finales()

    preparar >> staging_info >> transformacion >> metricas


elt_duckdb_pipeline_V2 = elt_pipeline()